# Option 2 — Symbolic Conditioned MIDI Generation

**Task**: Given a 4-second MIDI prefix, autoregressively generate a 4-second continuation using a small causal Transformer trained on MAESTRO v3.0.0.

**Pipeline**:
1. Setup & dependencies
2. Download MAESTRO MIDI-only dataset
3. Pre-cache piano-rolls (parse once, load fast)
4. Build sliding window index
5. Train causal Transformer (BCEWithLogitsLoss, teacher forcing)
6. Generate `symbolic_conditioned.mid`
7. Evaluate vs copy-last-frame baseline

In [1]:
!nvidia-smi

Sun May 24 07:38:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Environment Setup

In [2]:
import os, sys
from pathlib import Path

# Detect Colab
IN_COLAB = 'google.colab' in sys.modules
print('Running on Colab:', IN_COLAB)

if IN_COLAB:
    # Clone repo if not already present
    REPO_URL = 'https://github.com/archerop/CSE_253.git'  # update if different
    if not Path('/content/CSE_253').exists():
        !git clone {REPO_URL} /content/CSE_253
    ASSIGNMENT_ROOT = Path('/content/CSE_253/assignment2')
else:
    # Local: notebook lives inside assignment2/
    ASSIGNMENT_ROOT = Path('__file__').resolve().parent if '__file__' in dir() else Path('.').resolve()
    # Walk up until we find the app/ directory
    for p in [ASSIGNMENT_ROOT] + list(ASSIGNMENT_ROOT.parents):
        if (p / 'app').exists():
            ASSIGNMENT_ROOT = p
            break

print('Assignment root:', ASSIGNMENT_ROOT)
os.chdir(ASSIGNMENT_ROOT)
sys.path.insert(0, str(ASSIGNMENT_ROOT))

Running on Colab: True
Cloning into '/content/CSE_253'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 28 (delta 1), reused 28 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 102.38 KiB | 25.59 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Assignment root: /content/CSE_253/assignment2


In [3]:
# Install dependencies
import importlib

def need(pkg, import_name=None):
    return importlib.util.find_spec(import_name or pkg) is None

pkgs = []
if need('pretty_midi'):  pkgs.append('pretty_midi')
if need('torch'):        pkgs.append('torch')
if need('pandas'):       pkgs.append('pandas')
if need('tqdm'):         pkgs.append('tqdm')
if need('scipy'):        pkgs.append('scipy')

if pkgs:
    print('Installing:', pkgs)
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
else:
    print('All dependencies already installed.')

Installing: ['pretty_midi']


## 2. Download MAESTRO MIDI Dataset

In [4]:
import urllib.request, zipfile, shutil

MAESTRO_ROOT = ASSIGNMENT_ROOT / 'data' / 'maestro-v3.0.0'
MAESTRO_CSV  = MAESTRO_ROOT / 'maestro-v3.0.0.csv'

if MAESTRO_CSV.exists():
    print('MAESTRO already present at', MAESTRO_ROOT)
else:
    URL = 'https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip'
    ZIP = ASSIGNMENT_ROOT / 'data' / 'downloads' / 'maestro-v3.0.0-midi.zip'
    ZIP.parent.mkdir(parents=True, exist_ok=True)

    if not ZIP.exists():
        print('Downloading MAESTRO MIDI (~57 MB)...')
        urllib.request.urlretrieve(URL, ZIP)
        print('Download complete.')

    print('Extracting...')
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(ASSIGNMENT_ROOT / 'data')
    print('Done. MAESTRO root:', MAESTRO_ROOT)

midi_count = len(list(MAESTRO_ROOT.rglob('*.midi')))
print(f'MIDI files found: {midi_count}')

Download complete.
Extracting...
Done. MAESTRO root: /content/CSE_253/assignment2/data/maestro-v3.0.0
MIDI files found: 1276


## 3. Imports

In [ ]:
import math, json, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pretty_midi
from tqdm.auto import tqdm

from app.shared.config import (
    MAESTRO_ROOT, MIDI_LOW, MIDI_HIGH, N_PITCHES,
    OPTION2_CACHE_DIR, OPTION2_OUTPUT_DIR, CHECKPOINT_DIR,
    OPTION2_FRAME_RATE, OPTION2_PREFIX_SECONDS, OPTION2_CONTINUATION_SECONDS,
    OPTION2_STRIDE_SECONDS, OPTION2_D_MODEL, OPTION2_NHEAD, OPTION2_NUM_LAYERS,
    OPTION2_DIM_FEEDFORWARD, OPTION2_DROPOUT, OPTION2_BATCH_SIZE,
    OPTION2_LEARNING_RATE, OPTION2_WEIGHT_DECAY, OPTION2_MAX_EPOCHS, OPTION2_PATIENCE,
)
from app.shared.metadata import load_maestro_metadata, validate_maestro_paths
from app.option2.symbolic_dataset import (
    build_window_index, SymbolicDataset, precache_pianorolls, _midi_to_pianoroll
)
from app.option2.symbolic_models import SymbolicTransformer, CopyLastFrameBaseline
from app.option2.symbolic_train import train, load_best_checkpoint
from app.option2.symbolic_generate import (
    extract_prefix, generate_conditioned, save_symbolic_conditioned, pianoroll_to_midi
)
from app.option2.symbolic_eval import evaluate_generation, print_metrics

print('Imports OK')

ImportError: cannot import name 'OPTION2_FRAME_RATE' from 'app.shared.config' (/content/CSE_253/assignment2/app/shared/config.py)

In [ ]:
# Device selection
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print('Device:', DEVICE)

## 4. Pre-cache Piano-rolls
Parses all 1,276 MIDI files once and saves as `.npy`. Skips already-cached files, so safe to re-run.

In [ ]:
precache_pianorolls()

## 5. Build Window Index

In [ ]:
OPTION2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_windows = build_window_index(
    split='train',
    cache_path=OPTION2_CACHE_DIR / 'train_windows.pkl',
)
val_windows = build_window_index(
    split='validation',
    cache_path=OPTION2_CACHE_DIR / 'val_windows.pkl',
)
test_windows = build_window_index(
    split='test',
    cache_path=OPTION2_CACHE_DIR / 'test_windows.pkl',
)

print(f'Windows — train: {len(train_windows):,}  val: {len(val_windows):,}  test: {len(test_windows):,}')

## 6. Dataset & DataLoader

In [ ]:
train_ds = SymbolicDataset(train_windows)
val_ds   = SymbolicDataset(val_windows)

# Quick sanity check
prefix, cont = train_ds[0]
print(f'prefix shape: {tuple(prefix.shape)}  cont shape: {tuple(cont.shape)}')
print(f'prefix active notes/frame (mean): {prefix.sum(-1).mean():.2f}')
print(f'cont   active notes/frame (mean): {cont.sum(-1).mean():.2f}')

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
BATCH_SIZE  = OPTION2_BATCH_SIZE   # 32
NUM_WORKERS = 4 if DEVICE.type != 'cpu' else 2
PIN_MEMORY  = DEVICE.type == 'cuda'
# ────────────────────────────────────────────────────────────────────────────

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=max(1, NUM_WORKERS // 2), pin_memory=PIN_MEMORY
)

print(f'Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}')

## 7. Model

| Component | Value |
|-----------|-------|
| Architecture | Causal Transformer (GPT-style) |
| Input | 88-dim binary piano-roll frame |
| d_model | 128 |
| Layers | 4 |
| Heads | 4 |
| FFN dim | 512 |
| Loss | BCEWithLogitsLoss |
| Training | Teacher forcing over `[prefix \| continuation[:-1]]` |

In [ ]:
model = SymbolicTransformer().to(DEVICE)
print(f'Parameters: {model.count_parameters():,}')

## 8. Training

In [ ]:
CKPT_PATH = CHECKPOINT_DIR / 'option2_best.pt'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    checkpoint_path=CKPT_PATH,
)

OPTION2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OPTION2_OUTPUT_DIR / 'train_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Training complete. Checkpoint:', CKPT_PATH)

## 9. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], label='Train')
plt.plot(epochs, history['val_loss'],   label='Val')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Option 2 Training Loss')
plt.legend()
plt.tight_layout()
plt.savefig(OPTION2_OUTPUT_DIR / 'loss_curve.png', dpi=150)
plt.show()

## 10. Generate `symbolic_conditioned.mid`

In [ ]:
# Pick a test-split piece
df = load_maestro_metadata(MAESTRO_ROOT)
df = validate_maestro_paths(df)
test_df = df[(df['split'] == 'test') & df['midi_exists']].reset_index(drop=True)
test_df = test_df.sort_values('duration')
row = test_df.iloc[len(test_df) // 2]
MIDI_PATH = row['midi_path']
print(f"Using: {row['composer']} — {row['title']}  ({row['duration']:.1f}s)")
print(f"File:  {MIDI_PATH}")

In [ ]:
# Load best checkpoint
model = load_best_checkpoint(SymbolicTransformer(), CKPT_PATH, DEVICE)

# Generate
output_path = save_symbolic_conditioned(
    prefix_midi_path=MIDI_PATH,
    model=model,
    device=DEVICE,
)
print('Output saved to:', output_path)

## 11. Evaluation — Model vs Baseline

In [ ]:
from app.option2.symbolic_dataset import _load_roll

frame_rate = OPTION2_FRAME_RATE
prefix_len = int(OPTION2_PREFIX_SECONDS * frame_rate)
cont_len   = int(OPTION2_CONTINUATION_SECONDS * frame_rate)

roll = _load_roll(MIDI_PATH, frame_rate)
gt_roll   = roll[prefix_len : prefix_len + cont_len]

prefix_tensor = extract_prefix(MIDI_PATH)

# Model prediction
model_roll = generate_conditioned(model, prefix_tensor, cont_len, DEVICE)

# Baseline: copy last prefix frame
baseline   = CopyLastFrameBaseline()
baseline_roll = baseline(prefix_tensor.unsqueeze(0), cont_len).squeeze(0).numpy()

print('=== Model ===')
print_metrics(evaluate_generation(model_roll, gt_roll))

print('=== Baseline (copy last frame) ===')
print_metrics(evaluate_generation(baseline_roll, gt_roll))

In [ ]:
# Visualise prefix + generated continuation as a piano-roll plot
import matplotlib.pyplot as plt
import numpy as np

prefix_np = extract_prefix(MIDI_PATH).numpy()  # (P, 88)
combined  = np.concatenate([prefix_np, model_roll], axis=0).T  # (88, P+C)

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(combined, aspect='auto', origin='lower', cmap='Blues', interpolation='nearest')
ax.axvline(prefix_len - 0.5, color='red', linewidth=1.5, label='prefix | generated')
ax.set_xlabel('Frame (40 fps)')
ax.set_ylabel('Pitch (MIDI 21–108)')
ax.set_title('Piano-roll: prefix (left) + generated continuation (right)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(OPTION2_OUTPUT_DIR / 'pianoroll_visualization.png', dpi=150)
plt.show()

## Done

Outputs:
- `outputs/option2/symbolic_conditioned.mid` — final deliverable
- `outputs/option2/loss_curve.png` — training loss
- `outputs/option2/pianoroll_visualization.png` — piano-roll plot
- `outputs/checkpoints/option2_best.pt` — best model checkpoint